# Lecture 10 — Exercise: CO₂ Emissions Dashboard

**Goal:** Build a full interactive CO₂ Emissions Streamlit dashboard with 5 widget types, chained filters, guards, and two reactive charts.

> **How to run:** Copy the final cell into a `.py` file and run `streamlit run lecture10_exercise.py`
>
> Or run: `streamlit run week10/lecture10_exercise.py` from the repo root.

---
## Setup — Imports & Data Loading

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import datetime
from pathlib import Path

st.set_page_config(page_title="CO2 Dashboard", page_icon="🌱", layout="wide")

# @st.cache_data: prevents re-reading the CSV on every widget interaction
@st.cache_data
def load_data():
    path = Path(__file__).parent.parent / 'data' / 'co2_emissions.csv'
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')
    return df

df = load_data()

st.title("🌱 CO2 Emissions Explorer")
st.caption("Source: Our World in Data — ourworldindata.org/co2-emissions")

---
## Task 1 — Sidebar with 5 Widgets

| Widget | Purpose |
|---|---|
| `st.selectbox` | Region filter (includes 'All') |
| `st.multiselect` | Country filter (chained from region) |
| `st.date_input` | Date range (two-handle) |
| `st.radio` | Metric toggle: Total CO2 vs per capita |
| `st.checkbox` | Highlight top emitter only |

**Guards:**
- Incomplete date range → `st.warning` + `st.stop()`
- Empty country selection → `st.warning` + `st.stop()`

In [ ]:
with st.sidebar:
    st.header("Filters")

    # a) selectbox — Region with 'All' option
    regions = ['All'] + sorted(df['Region'].unique())
    selected_region = st.selectbox("Region", regions)

    # b) multiselect — Countries chained from region
    if selected_region == 'All':
        country_options = sorted(df['Country'].unique())
    else:
        country_options = sorted(df[df['Region'] == selected_region]['Country'].unique())

    selected_countries = st.multiselect(
        "Countries",
        options=country_options,
        default=country_options[:3]
    )

    # c) date_input — two-handle range; years stored as Jan-1 dates
    date_range = st.date_input(
        "Date range",
        value=(
            datetime.date(int(df['Year'].min()), 1, 1),
            datetime.date(int(df['Year'].max()), 1, 1)
        ),
        min_value=datetime.date(int(df['Year'].min()), 1, 1),
        max_value=datetime.date(int(df['Year'].max()), 1, 1),
        format="YYYY-MM-DD"
    )

    st.divider()

    # d) radio — Metric toggle (2–4 options → radio beats selectbox)
    metric = st.radio("Metric", ["Total CO2 (Mt)", "CO2 per capita"])

    # e) checkbox — highlight mode
    highlight_top = st.checkbox("Show only top emitter highlighted")

# ── Guards ────────────────────────────────────────────────────────────────────
# User may click start date but not end date yet
if len(date_range) != 2:
    st.warning("Select a start AND end date in the sidebar.")
    st.stop()

if not selected_countries:
    st.warning("Select at least one country in the sidebar.")
    st.stop()

# ── Convert date_input → pd.Timestamp before pandas comparisons ──────────────
start_ts = pd.Timestamp(date_range[0])
end_ts   = pd.Timestamp(date_range[1])

filtered = df[
    df['Country'].isin(selected_countries) &
    (df['Date'] >= start_ts) &
    (df['Date'] <= end_ts)
]

---
## Task 2 — Filter Summary Caption

**BBD rule:** Always show users how many records match current filters — reduces confusion when charts look sparse.

In [ ]:
year_start = date_range[0].year
year_end   = date_range[1].year

st.caption(
    f"{len(selected_countries)} countries | "
    f"{selected_region} | "
    f"{year_start}–{year_end} | "
    f"{metric}"
)

---
## Task 3 — Two Charts Reacting to All Filters

**Left — Line chart:** Metric over time, one line per country.  
When checkbox is on: grey all lines except the top emitter; label it at the end (SWD grey-and-highlight technique).

**Right — Bar chart:** Country ranking for the last year in the selected date range.

**BBD colour types (required comment):**
- Line chart with highlight: *categorical + highlight* colour
- Bar chart: *sequential single-hue* colour

In [ ]:
y_col   = 'CO2_Mt' if metric == "Total CO2 (Mt)" else 'CO2_per_capita'
y_label = 'CO2 Emissions (Mt)' if y_col == 'CO2_Mt' else 'CO2 per Capita (t)'

col_left, col_right = st.columns([2, 1])

# ── Left: Line chart ──────────────────────────────────────────────────────────
with col_left:
    if highlight_top:
        # SWD grey-and-highlight: find the top emitter by total emissions in range
        top_country = filtered.groupby('Country')[y_col].sum().idxmax()

        # Categorical + highlight colour: grey for all, accent for top
        color_map = {c: '#CCCCCC' for c in selected_countries}
        color_map[top_country] = '#E63946'

        fig_line = px.line(
            filtered, x='Date', y=y_col, color='Country',
            color_discrete_map=color_map,
            labels={y_col: y_label, 'Date': ''},
            title=f"Highest emitter: {top_country}"
        )

        # Label top emitter at the end of its line
        top_data = filtered[filtered['Country'] == top_country].sort_values('Date')
        if not top_data.empty:
            last_row = top_data.iloc[-1]
            fig_line.add_annotation(
                x=last_row['Date'],
                y=last_row[y_col],
                text=f"  {top_country}",
                showarrow=False,
                xanchor='left',
                font=dict(color='#E63946', size=11)
            )
    else:
        # Categorical colour: one distinct colour per country
        fig_line = px.line(
            filtered, x='Date', y=y_col, color='Country',
            labels={y_col: y_label, 'Date': ''},
            title=f"{metric} over time"
        )

    # SWD: white background
    fig_line.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(family='Arial')
    )
    st.plotly_chart(fig_line, use_container_width=True)

# ── Right: Bar chart — ranking for last year in selected range ────────────────
with col_right:
    last_year = int(filtered['Year'].max())
    latest    = filtered[filtered['Year'] == last_year].sort_values(y_col)

    # Sequential single-hue colour: uniform blue — ranking, not categories
    fig_bar = px.bar(
        latest, x=y_col, y='Country', orientation='h',
        color_discrete_sequence=['#2E75B6'],
        labels={y_col: y_label, 'Country': ''},
        title=f"Country ranking ({last_year})"
    )
    fig_bar.update_layout(
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(family='Arial'),
        xaxis=dict(range=[0, latest[y_col].max() * 1.15])
    )
    fig_bar.update_traces(marker_line_width=0)
    st.plotly_chart(fig_bar, use_container_width=True)

---
## Extension — KPI Row Above the Charts

Three metric cards:
1. Total CO₂ in last year of selected range (sum across selected countries)
2. % change from first to last year
3. Country with highest emissions in last year

In [ ]:
# Note: place this block BEFORE the col_left / col_right columns block above

first_year     = int(filtered['Year'].min())
last_year_data = filtered[filtered['Year'] == last_year]
first_year_data= filtered[filtered['Year'] == first_year]

total_last  = last_year_data[y_col].sum()
total_first = first_year_data[y_col].sum()
pct_change  = ((total_last - total_first) / total_first * 100) if total_first != 0 else 0
top_emitter = (
    last_year_data.loc[last_year_data[y_col].idxmax(), 'Country']
    if not last_year_data.empty else 'N/A'
)

kpi1, kpi2, kpi3 = st.columns(3)

with kpi1:
    unit = 'Mt' if y_col == 'CO2_Mt' else 't'
    st.metric(f"Total {metric} ({last_year})", f"{total_last:,.1f} {unit}")

with kpi2:
    st.metric(
        f"Change {first_year} → {last_year}",
        f"{pct_change:+.1f}%"
    )

with kpi3:
    st.metric(f"Top Emitter ({last_year})", top_emitter)

---
## Complete Solution — Copy this into `lecture10_exercise.py` and run with Streamlit

```bash
streamlit run week10/lecture10_exercise.py
```

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import datetime
from pathlib import Path

st.set_page_config(page_title="CO2 Dashboard", page_icon="🌱", layout="wide")

# ── Data ──────────────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    path = Path(__file__).parent.parent / 'data' / 'co2_emissions.csv'
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-01-01')
    return df

df = load_data()

st.title("🌱 CO2 Emissions Explorer")
st.caption("Source: Our World in Data — ourworldindata.org/co2-emissions")

# ── TASK 1: Sidebar with 5 widgets ────────────────────────────────────────────
with st.sidebar:
    st.header("Filters")

    # a) selectbox — Region
    regions = ['All'] + sorted(df['Region'].unique())
    selected_region = st.selectbox("Region", regions)

    # b) multiselect — Countries (chained from region)
    if selected_region == 'All':
        country_options = sorted(df['Country'].unique())
    else:
        country_options = sorted(df[df['Region'] == selected_region]['Country'].unique())

    selected_countries = st.multiselect(
        "Countries",
        options=country_options,
        default=country_options[:3]
    )

    # c) date_input — two-handle date range
    date_range = st.date_input(
        "Date range",
        value=(
            datetime.date(int(df['Year'].min()), 1, 1),
            datetime.date(int(df['Year'].max()), 1, 1)
        ),
        min_value=datetime.date(int(df['Year'].min()), 1, 1),
        max_value=datetime.date(int(df['Year'].max()), 1, 1),
        format="YYYY-MM-DD"
    )

    st.divider()

    # d) radio — Metric
    metric = st.radio("Metric", ["Total CO2 (Mt)", "CO2 per capita"])

    # e) checkbox — highlight toggle
    highlight_top = st.checkbox("Show only top emitter highlighted")

# ── Guards ────────────────────────────────────────────────────────────────────
if len(date_range) != 2:
    st.warning("Select a start AND end date in the sidebar.")
    st.stop()

if not selected_countries:
    st.warning("Select at least one country in the sidebar.")
    st.stop()

# ── Filter data ───────────────────────────────────────────────────────────────
start_ts = pd.Timestamp(date_range[0])
end_ts   = pd.Timestamp(date_range[1])

filtered = df[
    df['Country'].isin(selected_countries) &
    (df['Date'] >= start_ts) &
    (df['Date'] <= end_ts)
]

# ── TASK 2: Filter summary caption ────────────────────────────────────────────
year_start = date_range[0].year
year_end   = date_range[1].year
st.caption(
    f"{len(selected_countries)} countries | "
    f"{selected_region} | "
    f"{year_start}–{year_end} | "
    f"{metric}"
)

# ── EXTENSION: KPI row ────────────────────────────────────────────────────────
y_col   = 'CO2_Mt' if metric == "Total CO2 (Mt)" else 'CO2_per_capita'
y_label = 'CO2 Emissions (Mt)' if y_col == 'CO2_Mt' else 'CO2 per Capita (t)'

last_year      = int(filtered['Year'].max())
first_year     = int(filtered['Year'].min())
last_year_data = filtered[filtered['Year'] == last_year]
first_year_data= filtered[filtered['Year'] == first_year]

total_last  = last_year_data[y_col].sum()
total_first = first_year_data[y_col].sum()
pct_change  = ((total_last - total_first) / total_first * 100) if total_first != 0 else 0
top_emitter = (
    last_year_data.loc[last_year_data[y_col].idxmax(), 'Country']
    if not last_year_data.empty else 'N/A'
)

kpi1, kpi2, kpi3 = st.columns(3)
unit = 'Mt' if y_col == 'CO2_Mt' else 't'
with kpi1:
    st.metric(f"Total {metric} ({last_year})", f"{total_last:,.1f} {unit}")
with kpi2:
    st.metric(f"Change {first_year} → {last_year}", f"{pct_change:+.1f}%")
with kpi3:
    st.metric(f"Top Emitter ({last_year})", top_emitter)

# ── TASK 3: Two charts ────────────────────────────────────────────────────────
col_left, col_right = st.columns([2, 1])

with col_left:
    if highlight_top:
        # Categorical + highlight colour (BBD)
        top_country = filtered.groupby('Country')[y_col].sum().idxmax()
        color_map = {c: '#CCCCCC' for c in selected_countries}
        color_map[top_country] = '#E63946'

        fig_line = px.line(
            filtered, x='Date', y=y_col, color='Country',
            color_discrete_map=color_map,
            labels={y_col: y_label, 'Date': ''},
            title=f"Highest emitter: {top_country}"
        )

        top_data = filtered[filtered['Country'] == top_country].sort_values('Date')
        if not top_data.empty:
            last_row = top_data.iloc[-1]
            fig_line.add_annotation(
                x=last_row['Date'], y=last_row[y_col],
                text=f"  {top_country}",
                showarrow=False, xanchor='left',
                font=dict(color='#E63946', size=11)
            )
    else:
        # Categorical colour (BBD)
        fig_line = px.line(
            filtered, x='Date', y=y_col, color='Country',
            labels={y_col: y_label, 'Date': ''},
            title=f"{metric} over time"
        )

    fig_line.update_layout(
        plot_bgcolor='white', paper_bgcolor='white', font=dict(family='Arial')
    )
    st.plotly_chart(fig_line, use_container_width=True)

with col_right:
    latest = filtered[filtered['Year'] == last_year].sort_values(y_col)

    # Sequential single-hue colour (BBD)
    fig_bar = px.bar(
        latest, x=y_col, y='Country', orientation='h',
        color_discrete_sequence=['#2E75B6'],
        labels={y_col: y_label, 'Country': ''},
        title=f"Country ranking ({last_year})"
    )
    fig_bar.update_layout(
        plot_bgcolor='white', paper_bgcolor='white',
        font=dict(family='Arial'),
        xaxis=dict(range=[0, latest[y_col].max() * 1.15])
    )
    fig_bar.update_traces(marker_line_width=0)
    st.plotly_chart(fig_bar, use_container_width=True)